In [6]:
# ==============================================================================
# SCRIPT TRAINING: MENGGUNAKAN DATA CUPLIKAN 50 MB
# ==============================================================================

# --- 0. IMPOR LIBRARY YANG DIBUTUHKAN ---
import pandas as pd
import numpy as np
from tqdm.auto import tqdm

from sklearn.model_selection import KFold
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error

print("Semua library berhasil diimpor!")

# --- 1. MEMUAT DATA SUPER RINGKAS KITA ---
# Pastikan path file ini benar
file_path = 'C:/KULIAH STATISTIKA UNIVERSITAS SYIAH KUALA/KEGIATAN/Dataquest/train_cuplikan_final.csv'

print(f"\nMemuat data dari: {file_path}")
df = pd.read_csv(file_path)

# Jaga-jaga jika ada baris teks yang kosong (NaN) setelah dimuat
df.fillna('', inplace=True)
print(f"Data cuplikan berhasil dimuat. Jumlah baris: {len(df)}")


# --- 2. PERSIAPAN FINAL SEBELUM TRAINING ---

print(f"\nJumlah data sebelum membuang outlier: {len(df)}")
df_train = df[df['lama hukuman (bulan)'] != 88888].copy()
print(f"Jumlah data setelah membuang outlier: {len(df_train)}")

# Kolom 'teks_bersih' sudah siap dari proses sebelumnya
X = df_train['teks_bersih']
y = df_train['lama hukuman (bulan)']


# --- 3. MEMBANGUN PIPELINE DENGAN RANDOM FOREST ---

pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(max_features=10000, ngram_range=(1, 2))),
    ('model', RandomForestRegressor(random_state=42, n_jobs=-1))
])
print("\nPipeline model dengan Random Forest berhasil dibuat!")


# --- 4. MELATIH & EVALUASI DENGAN PROGRESS BAR ---

print("Memulai evaluasi model dengan 5-Fold Cross-Validation...")
kf = KFold(n_splits=5, shuffle=True, random_state=42)
rmse_scores = []

for train_index, val_index in tqdm(kf.split(X), total=kf.get_n_splits(), desc="Cross-Validation Folds"):
    X_train, X_val = X.iloc[train_index], X.iloc[val_index]
    y_train, y_val = y.iloc[train_index], y.iloc[val_index]
    
    pipeline.fit(X_train, y_train)
    predictions = pipeline.predict(X_val)
    fold_rmse = np.sqrt(mean_squared_error(y_val, predictions))
    rmse_scores.append(fold_rmse)

print("\n--- HASIL EVALUASI AKHIR ---")
print(f"Skor RMSE untuk setiap fold: {np.round(rmse_scores, 2)}")
print(f"Rata-rata RMSE: {np.mean(rmse_scores):.2f} bulan")
print(f"Standar Deviasi RMSE: {np.std(rmse_scores):.2f} bulan")
print("----------------------------")

print("\nProses training dan evaluasi selesai!")

Semua library berhasil diimpor!

Memuat data dari: C:/KULIAH STATISTIKA UNIVERSITAS SYIAH KUALA/KEGIATAN/Dataquest/train_cuplikan_final.csv
Data cuplikan berhasil dimuat. Jumlah baris: 16572

Jumlah data sebelum membuang outlier: 16572
Jumlah data setelah membuang outlier: 16571

Pipeline model dengan Random Forest berhasil dibuat!
Memulai evaluasi model dengan 5-Fold Cross-Validation...


Cross-Validation Folds: 100%|██████████| 5/5 [47:16<00:00, 567.29s/it]


--- HASIL EVALUASI AKHIR ---
Skor RMSE untuk setiap fold: [22.53 22.6  23.35 22.39 22.13]
Rata-rata RMSE: 22.60 bulan
Standar Deviasi RMSE: 0.41 bulan
----------------------------

Proses training dan evaluasi selesai!


In [7]:
# ==============================================================================
# CELL 3: TESTING & PEMBUATAN FILE SUBMISI
# ==============================================================================

# Definisikan lokasi file test
test_csv_path = 'C:/KULIAH STATISTIKA UNIVERSITAS SYIAH KUALA/KEGIATAN/Dataquest/objective-quest-2025/test.csv'
# Folder file putusan tetap sama
text_files_folder = 'C:/KULIAH STATISTIKA UNIVERSITAS SYIAH KUALA/KEGIATAN/Dataquest/objective-quest-2025/file_putusan/'

print(f"Memuat data test dari: {test_csv_path}")
df_test = pd.read_csv(test_csv_path)

# --- Proses Pembersihan Data Test (Gunakan Resep yang Sama Persis) ---
print("\nMemulai proses pembersihan pada data test...")
list_teks_putusan_test = []

# Kita menggunakan fungsi-fungsi yang sudah didefinisikan di cell pertama
# (bersihkan_header_footer_v3, ambil_cuplikan_penting)
for index, row in tqdm(df_test.iterrows(), total=len(df_test), desc="Mengolah file teks test"):
    doc_id = row['id']
    file_path = os.path.join(text_files_folder, f"{doc_id}.txt")
    try:
        with open(file_path, 'r', encoding='utf-8') as file:
            isi_teks_mentah = file.read()
            teks_tanpa_header = bersihkan_header_footer_v3(isi_teks_mentah)
            cuplikan_teks = ambil_cuplikan_penting(teks_tanpa_header)
            list_teks_putusan_test.append(cuplikan_teks)
    except Exception as e:
        print(f"ERROR pada id {doc_id}: {e}")
        list_teks_putusan_test.append("")

df_test['teks_putusan'] = list_teks_putusan_test
df_test['teks_putusan'].fillna('', inplace=True)

# Terapkan pembersihan tuntas
df_test['teks_bersih'] = df_test['teks_putusan'].apply(pembersihan_tuntas)
print("Data test berhasil diproses.")


# --- Membuat Prediksi dengan Model Final ---
X_test = df_test['teks_bersih']
print("\nMembuat prediksi pada data test...")
# Gunakan pipeline yang SUDAH DILATIH dari cell sebelumnya
predictions = pipeline.predict(X_test)
print("Prediksi selesai.")


# --- Poles Hasil & Buat File Submisi ---
# Bulatkan hasil prediksi dan pastikan tidak ada nilai negatif
final_predictions = np.round(predictions)
final_predictions[final_predictions < 0] = 0

# Buat DataFrame untuk submisi sesuai format
submission_df = pd.DataFrame({
    'id': df_test['id'],
    'lama hukuman (bulan)': final_predictions.astype(int)
})

# Simpan ke file submission.csv
submission_path = 'C:/KULIAH STATISTIKA UNIVERSITAS SYIAH KUALA/KEGIATAN/Dataquest/submission.csv'
submission_df.to_csv(submission_path, index=False)

print(f"\nFile submisi berhasil dibuat di: {submission_path}")
print("Contoh isi file submisi:")
print(submission_df.head())
print("\nProses selesai! Semoga berhasil! 🏆")

Memuat data test dari: C:/KULIAH STATISTIKA UNIVERSITAS SYIAH KUALA/KEGIATAN/Dataquest/objective-quest-2025/test.csv

Memulai proses pembersihan pada data test...


Mengolah file teks test: 100%|██████████| 6666/6666 [00:14<00:00, 469.16it/s]
C:\Users\ASUS\AppData\Local\Temp\ipykernel_24140\4211879094.py:33: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df_test['teks_putusan'].fillna('', inplace=True)


Data test berhasil diproses.

Membuat prediksi pada data test...
Prediksi selesai.

File submisi berhasil dibuat di: C:/KULIAH STATISTIKA UNIVERSITAS SYIAH KUALA/KEGIATAN/Dataquest/submission.csv
Contoh isi file submisi:
          id  lama hukuman (bulan)
0   doc_7650                    19
1  doc_22122                   104
2  doc_13812                    74
3  doc_16571                    16
4   doc_9900                    10

Proses selesai! Semoga berhasil! 🏆


In [2]:
import pandas as pd
import os
import re
import time
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
import lightgbm as lgb

# =============================================================================
# 1. KONFIGURASI
# =============================================================================
DATA_PATH = 'objective-quest-2025/file_putusan/'
TRAIN_CSV_PATH = 'train.csv'
RANDOM_STATE = 42

# =============================================================================
# 2. FUNGSI-FUNGSI HELPER (DENGAN MODIFIKASI)
# =============================================================================

def clean_text_safely(text):
    """
    Fungsi pembersih teks yang paling aman, memfilter baris demi baris,
    DAN DITAMBAH PENGHAPUSAN ANGKA/SIMBOL.
    """
    # Mengubah ke huruf kecil di awal agar pencarian pola konsisten
    text = text.lower()
    
    # Daftar pola regex untuk mengidentifikasi baris boilerplate yang harus dibuang
    noise_patterns = [
        re.compile(r'^\s*hkama.*$'),
        re.compile(r'^\s*ahkamah agung repub.*$'),
        re.compile(r'^\s*mah agung republik indonesia.*$'),
        re.compile(r'^\s*blik indonesi.*$'),
        re.compile(r'^\s*direktori putusan.*$'),
        re.compile(r'^\s*disclaimer\s*$'),
        re.compile(r'^\s*kepaniteraan mahkamah agung.*$'),
        re.compile(r'^\s*halaman \d+ dari \d+.*$'),
        re.compile(r'.*\(ext\.\d+\)$')
    ]
    
    lines = text.splitlines()
    cleaned_lines = []
    for line in lines:
        is_noise = False
        for pattern in noise_patterns:
            if pattern.search(line):
                is_noise = True
                break
        if not is_noise:
            cleaned_lines.append(line)
            
    # Gabungkan kembali baris-baris yang sudah bersih dari boilerplate
    cleaned_text = "\n".join(cleaned_lines)
    
    # *** SINTAKS TAMBAHAN YANG ANDA MINTA ***
    # Menghapus semua karakter kecuali huruf dan spasi
    cleaned_text = re.sub(r'[^a-zA-Z\s]', '', cleaned_text)
    # Menormalkan semua jenis spasi menjadi satu spasi tunggal
    cleaned_text = re.sub(r'\s+', ' ', cleaned_text).strip()
    
    return cleaned_text

def get_cleaned_text_robust(doc_id):
    """
    Membaca dan membersihkan teks menggunakan fungsi pembersih yang paling aman.
    (Tidak ada perubahan di sini, karena perubahannya ada di dalam `clean_text_safely`)
    """
    file_path = os.path.join(DATA_PATH, f'{doc_id}.txt')
    try:
        with open(file_path, 'r', encoding='utf-8', errors='ignore') as file:
            raw_text = file.read()
            cleaned_text = clean_text_safely(raw_text)
            return cleaned_text
    except FileNotFoundError:
        return ""

# =============================================================================
# 3. EKSEKUSI UTAMA
# =============================================================================
# Sisa dari skrip ini persis sama dengan yang Anda berikan.
if __name__ == '__main__':
    start_time = time.time()
    
    print("Membaca file train.csv...")
    df_train = pd.read_csv(TRAIN_CSV_PATH)
    
    print("Memulai proses pembersihan teks (termasuk penghapusan angka)...")
    df_train['teks'] = df_train['id'].apply(get_cleaned_text_robust)
    df_train.dropna(subset=['teks'], inplace=True)
    df_train = df_train[df_train['teks'].str.len() > 0].copy()
    print("✅ Teks berhasil dimuat dan dibersihkan.")

    print("\n--- Memproses kolom target 'lama hukuman (bulan)' ---")
    df_train = df_train[df_train['lama hukuman (bulan)'] != 88888].copy()
    print(f"✅ Baris outlier telah dihapus.")

    print("\n--- Mempersiapkan data untuk modeling ---")
    X = df_train['teks']
    y = df_train['lama hukuman (bulan)']
    
    X_train, X_val, y_train, y_val = train_test_split(
        X, y, test_size=0.2, random_state=RANDOM_STATE
    )
    print(f"Data dibagi menjadi {len(X_train)} baris training dan {len(X_val)} baris validasi.")

    print("\n--- Melakukan vektorisasi teks dengan TF-IDF ---")
    vectorizer = TfidfVectorizer(max_features=15000, min_df=3, ngram_range=(1, 2))
    
    X_train_tfidf = vectorizer.fit_transform(X_train)
    X_val_tfidf = vectorizer.transform(X_val)
    print(f"✅ Teks telah diubah menjadi matriks dengan {X_train_tfidf.shape[1]} fitur.")

    print("\n--- Melatih model baseline LightGBM ---")
    lgb_params = {
        'objective': 'rmse', 'metric': 'rmse', 'n_estimators': 1000,
        'learning_rate': 0.05, 'feature_fraction': 0.8, 'bagging_fraction': 0.8,
        'bagging_freq': 1, 'verbose': -1, 'n_jobs': -1, 'seed': RANDOM_STATE
    }
    
    model = lgb.LGBMRegressor(**lgb_params)
    
    model.fit(X_train_tfidf, y_train,
              eval_set=[(X_val_tfidf, y_val)],
              eval_metric='rmse',
              callbacks=[lgb.early_stopping(100, verbose=True)])

    print("✅ Model baseline berhasil dilatih.")
    
    print("\n--- Mengevaluasi performa model ---")
    predictions = model.predict(X_val_tfidf)
    
    rmse = np.sqrt(mean_squared_error(y_val, predictions))
    
    print("\n==============================================")
    print(f" 🎯 SKOR RMSE BASELINE: {rmse:.4f}")
    print("==============================================")
    
    end_time = time.time()
    print(f"\nTotal waktu eksekusi: {end_time - start_time:.2f} detik.")

Membaca file train.csv...
Memulai proses pembersihan teks (termasuk penghapusan angka)...
✅ Teks berhasil dimuat dan dibersihkan.

--- Memproses kolom target 'lama hukuman (bulan)' ---
✅ Baris outlier telah dihapus.

--- Mempersiapkan data untuk modeling ---
Data dibagi menjadi 13254 baris training dan 3314 baris validasi.

--- Melakukan vektorisasi teks dengan TF-IDF ---
✅ Teks telah diubah menjadi matriks dengan 15000 fitur.

--- Melatih model baseline LightGBM ---
Training until validation scores don't improve for 100 rounds
Did not meet early stopping. Best iteration is:
[978]	valid_0's rmse: 16.6072
✅ Model baseline berhasil dilatih.

--- Mengevaluasi performa model ---


c:\KULIAH STATISTIKA UNIVERSITAS SYIAH KUALA\Envi_3.10.8\lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(



 🎯 SKOR RMSE BASELINE: 16.6072

Total waktu eksekusi: 845.18 detik.


In [3]:
import pandas as pd
import os
import re
import time
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
import lightgbm as lgb

# =============================================================================
# 1. KONFIGURASI
# =============================================================================
DATA_PATH = 'objective-quest-2025/file_putusan/'
TRAIN_CSV_PATH = 'train.csv'
RANDOM_STATE = 42

# =============================================================================
# 2. FUNGSI-FUNGSI HELPER (DENGAN MODIFIKASI EKSPERIMEN)
# =============================================================================

def clean_text_experiment(text):
    """
    Fungsi pembersih EKSPERIMENTAL:
    - Mengubah ke huruf kecil.
    - Menghapus SEMUA karakter kecuali huruf dan spasi.
    - Menormalkan spasi.
    """
    # Mengubah ke huruf kecil
    text = text.lower()
    
    # Menghapus semua angka, simbol, dan tanda baca
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    
    # Menormalkan spasi
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text

def get_cleaned_text_robust(doc_id):
    """
    Membaca dan membersihkan teks menggunakan fungsi EKSPERIMENTAL.
    """
    file_path = os.path.join(DATA_PATH, f'{doc_id}.txt')
    try:
        with open(file_path, 'r', encoding='utf-8', errors='ignore') as file:
            raw_text = file.read()
            # Menggunakan fungsi pembersih baru untuk eksperimen ini
            cleaned_text = clean_text_experiment(raw_text)
            return cleaned_text
    except FileNotFoundError:
        return ""

# =============================================================================
# 3. EKSEKUSI UTAMA (Menggunakan Pipeline Terbaik Kita Sejauh Ini)
# =============================================================================

if __name__ == '__main__':
    start_time = time.time()
    
    # --- Tahap A: Memuat dan Membersihkan Data ---
    print("Membaca file train.csv...")
    df_train = pd.read_csv(TRAIN_CSV_PATH)
    
    print("Memulai proses pembersihan teks (EKSPERIMEN HAPUS ANGKA & SIMBOL)...")
    df_train['teks'] = df_train['id'].apply(get_cleaned_text_robust)
    df_train.dropna(subset=['teks'], inplace=True)
    df_train = df_train[df_train['teks'].str.len() > 0].copy()
    print("✅ Teks berhasil dimuat dan dibersihkan.")

    # --- Tahap B: Memproses Target Variabel (Menghapus Outlier) ---
    print("\n--- Memproses kolom target 'lama hukuman (bulan)' ---")
    df_train = df_train[df_train['lama hukuman (bulan)'] != 88888].copy()
    print(f"✅ Baris outlier telah dihapus.")

    # --- Tahap C: Persiapan untuk Modeling ---
    X = df_train['teks']
    y = df_train['lama hukuman (bulan)']
    
    X_train, X_val, y_train, y_val = train_test_split(
        X, y, test_size=0.2, random_state=RANDOM_STATE
    )
    print(f"\nData dibagi menjadi {len(X_train)} baris training dan {len(X_val)} baris validasi.")

    # --- Tahap D: Vektorisasi Teks (TF-IDF) dengan parameter optimal ---
    print("\n--- Melakukan vektorisasi teks dengan TF-IDF ---")
    vectorizer = TfidfVectorizer(max_features=15000, min_df=3, ngram_range=(1, 2))
    
    X_train_tfidf = vectorizer.fit_transform(X_train)
    X_val_tfidf = vectorizer.transform(X_val)
    print(f"✅ Teks telah diubah menjadi matriks dengan {X_train_tfidf.shape[1]} fitur.")

    # --- Tahap E: Training Model ---
    print("\n--- Melatih model LightGBM ---")
    lgb_params = {
        'objective': 'rmse', 'metric': 'rmse', 'n_estimators': 1000,
        'learning_rate': 0.05, 'feature_fraction': 0.8, 'bagging_fraction': 0.8,
        'bagging_freq': 1, 'verbose': -1, 'n_jobs': -1, 'seed': RANDOM_STATE
    }
    
    model = lgb.LGBMRegressor(**lgb_params)
    
    model.fit(X_train_tfidf, y_train,
              eval_set=[(X_val_tfidf, y_val)],
              eval_metric='rmse',
              callbacks=[lgb.early_stopping(100, verbose=False)])

    print("✅ Model berhasil dilatih.")
    
    # --- Tahap F: Evaluasi Model ---
    print("\n--- Mengevaluasi performa model ---")
    predictions = model.predict(X_val_tfidf)
    
    rmse = np.sqrt(mean_squared_error(y_val, predictions))
    
    print("\n==========================================================")
    print(f" 🎯 SKOR RMSE (EKSPERIMEN HAPUS ANGKA): {rmse:.4f}")
    print("==========================================================")
    
    end_time = time.time()
    print(f"\nTotal waktu eksekusi: {end_time - start_time:.2f} detik.")

Membaca file train.csv...
Memulai proses pembersihan teks (EKSPERIMEN HAPUS ANGKA & SIMBOL)...
✅ Teks berhasil dimuat dan dibersihkan.

--- Memproses kolom target 'lama hukuman (bulan)' ---
✅ Baris outlier telah dihapus.

Data dibagi menjadi 13256 baris training dan 3315 baris validasi.

--- Melakukan vektorisasi teks dengan TF-IDF ---
✅ Teks telah diubah menjadi matriks dengan 15000 fitur.

--- Melatih model LightGBM ---
✅ Model berhasil dilatih.

--- Mengevaluasi performa model ---


c:\KULIAH STATISTIKA UNIVERSITAS SYIAH KUALA\Envi_3.10.8\lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(



 🎯 SKOR RMSE (EKSPERIMEN HAPUS ANGKA): 16.6121

Total waktu eksekusi: 790.12 detik.


In [4]:
# =============================================================================
# LANGKAH 0: INSTALASI & IMPORT
# =============================================================================
# !pip install xgboost lightgbm scikit-learn pandas --quiet
import pandas as pd
import os
import re
import time
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Ridge
import lightgbm as lgb
import xgboost as xgb

# =============================================================================
# 1. KONFIGURASI
# =============================================================================
DATA_PATH = 'objective-quest-2025/file_putusan/'
TRAIN_CSV_PATH = 'train.csv'
RANDOM_STATE = 42

# =============================================================================
# 2. FUNGSI-FUNGSI HELPER
# =============================================================================
def clean_text_experiment(text):
    text = text.lower()
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def get_cleaned_text_robust(doc_id):
    file_path = os.path.join(DATA_PATH, f'{doc_id}.txt')
    try:
        with open(file_path, 'r', encoding='utf-8', errors='ignore') as file:
            raw_text = file.read()
            cleaned_text = clean_text_experiment(raw_text)
            return cleaned_text
    except FileNotFoundError:
        return ""

# =============================================================================
# 3. EKSEKUSI UTAMA - KOMPETISI MODEL
# =============================================================================
if __name__ == '__main__':
    print("Mempersiapkan data...")
    df_train = pd.read_csv(TRAIN_CSV_PATH)
    df_train['teks'] = df_train['id'].apply(get_cleaned_text_robust)
    df_train.dropna(subset=['teks'], inplace=True)
    df_train = df_train[df_train['teks'].str.len() > 0].copy()
    df_train = df_train[df_train['lama hukuman (bulan)'] != 88888].copy()
    
    X = df_train['teks']
    y = df_train['lama hukuman (bulan)']

    X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE)
    
    print("Melakukan vektorisasi teks dengan TF-IDF...")
    vectorizer = TfidfVectorizer(max_features=15000, min_df=3, ngram_range=(1, 2))
    X_train_tfidf = vectorizer.fit_transform(X_train)
    X_val_tfidf = vectorizer.transform(X_val)
    print("✅ Data fitur siap.")

    # --- Tahap D: Mendefinisikan Model-model yang Akan Diuji ---
    # *** PERUBAHAN DI SINI ***
    models_to_test = {
        "LightGBM": lgb.LGBMRegressor(
            objective='rmse', 
            random_state=RANDOM_STATE, 
            n_jobs=-1,
            n_estimators=1000, 
            learning_rate=0.05
        ),
        "XGBoost": xgb.XGBRegressor(
            objective='reg:squarederror', 
            random_state=RANDOM_STATE, 
            n_jobs=-1,
            n_estimators=1000, 
            learning_rate=0.05,
            early_stopping_rounds=100 # Parameter dipindah ke sini
        ),
        "RandomForest": RandomForestRegressor(
            random_state=RANDOM_STATE, 
            n_jobs=-1, 
            n_estimators=100, 
            max_depth=20 # Sedikit lebih dalam untuk perbandingan yang lebih adil
        ),
        "Ridge": Ridge(random_state=RANDOM_STATE)
    }

    results = []
    
    print("\n--- Memulai Kompetisi Model ---")
    for name, model in models_to_test.items():
        print(f"\nMelatih model: {name}...")
        start_iter_time = time.time()
        
        # *** PERBAIKAN LOGIKA FIT DI SINI ***
        if name == "LightGBM":
            model.fit(X_train_tfidf, y_train,
                      eval_set=[(X_val_tfidf, y_val)],
                      callbacks=[lgb.early_stopping(100, verbose=False)])
        elif name == "XGBoost":
            # Sekarang fit-nya lebih sederhana, karena parameter sudah diatur
            model.fit(X_train_tfidf, y_train,
                      eval_set=[(X_val_tfidf, y_val)],
                      verbose=False)
        else:
            model.fit(X_train_tfidf, y_train)

        predictions = model.predict(X_val_tfidf)
        rmse = np.sqrt(mean_squared_error(y_val, predictions))
        
        end_iter_time = time.time()
        duration = end_iter_time - start_iter_time
        
        results.append({'Model': name, 'RMSE': rmse, 'Waktu (detik)': duration})
        print(f"✅ {name} selesai dalam {duration:.2f} detik. RMSE: {rmse:.4f}")

    # --- Tampilkan Hasil Akhir ---
    print("\n\n==============================================")
    print("          HASIL AKHIR KOMPETISI MODEL")
    print("==============================================")
    results_df = pd.DataFrame(results)
    print(results_df.sort_values(by='RMSE').to_string(index=False))
    print("==============================================")

Mempersiapkan data...
Melakukan vektorisasi teks dengan TF-IDF...
✅ Data fitur siap.

--- Memulai Kompetisi Model ---

Melatih model: LightGBM...


c:\KULIAH STATISTIKA UNIVERSITAS SYIAH KUALA\Envi_3.10.8\lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


✅ LightGBM selesai dalam 471.83 detik. RMSE: 16.5242

Melatih model: XGBoost...
✅ XGBoost selesai dalam 1952.51 detik. RMSE: 16.7666

Melatih model: RandomForest...
✅ RandomForest selesai dalam 1297.53 detik. RMSE: 17.5362

Melatih model: Ridge...
✅ Ridge selesai dalam 2.83 detik. RMSE: 21.8403


          HASIL AKHIR KOMPETISI MODEL
       Model      RMSE  Waktu (detik)
    LightGBM 16.524226     471.827887
     XGBoost 16.766555    1952.511532
RandomForest 17.536239    1297.532087
       Ridge 21.840333       2.834168


In [5]:
import pandas as pd
import os
import re
import time
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
import lightgbm as lgb

# =============================================================================
# 1. KONFIGURASI
# =============================================================================
# Pastikan path ini sesuai dengan struktur folder Anda
DATA_PATH = 'objective-quest-2025/file_putusan/'
TRAIN_CSV_PATH = 'train.csv'
# Seed untuk reproduktibilitas
RANDOM_STATE = 42

# =============================================================================
# 2. FUNGSI-FUNGSI HELPER
# =============================================================================
def clean_text_experiment(text):
    """Membersihkan teks dengan menghapus angka dan simbol, lalu mengubah ke lowercase."""
    text = text.lower()
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def get_cleaned_text_robust(doc_id):
    """Membaca dan membersihkan teks dari file .txt."""
    file_path = os.path.join(DATA_PATH, f'{doc_id}.txt')
    try:
        with open(file_path, 'r', encoding='utf-8', errors='ignore') as file:
            raw_text = file.read()
            cleaned_text = clean_text_experiment(raw_text)
            return cleaned_text
    except FileNotFoundError:
        return ""

# =============================================================================
# 3. EKSEKUSI UTAMA - FOKUS PADA MODEL TERBAIK
# =============================================================================
if __name__ == '__main__':
    start_time = time.time()
    
    # --- Tahap A: Memuat dan Membersihkan Data ---
    print("Mempersiapkan data...")
    df_train = pd.read_csv(TRAIN_CSV_PATH)
    df_train['teks'] = df_train['id'].apply(get_cleaned_text_robust)
    df_train.dropna(subset=['teks'], inplace=True)
    df_train = df_train[df_train['teks'].str.len() > 0].copy()
    df_train = df_train[df_train['lama hukuman (bulan)'] != 88888].copy()
    
    X = df_train['teks']
    y = df_train['lama hukuman (bulan)']

    X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE)
    
    # --- Tahap B: Vektorisasi Teks dengan Parameter Optimal ---
    print("Melakukan vektorisasi teks dengan TF-IDF...")
    vectorizer = TfidfVectorizer(max_features=15000, min_df=3, ngram_range=(1, 2))
    X_train_tfidf = vectorizer.fit_transform(X_train)
    X_val_tfidf = vectorizer.transform(X_val)
    print("✅ Data fitur siap.")

    # --- Tahap C: Melatih Model Final (LightGBM) ---
    print("\n--- Melatih model LightGBM ---")
    
    lgb_params = {
        'objective': 'rmse', 
        'metric': 'rmse',
        'random_state': RANDOM_STATE, 
        'n_jobs': -1,
        'n_estimators': 1000, 
        'learning_rate': 0.05
    }
    
    model = lgb.LGBMRegressor(**lgb_params)
    
    model.fit(X_train_tfidf, y_train,
              eval_set=[(X_val_tfidf, y_val)],
              callbacks=[lgb.early_stopping(100, verbose=False)])

    print("✅ Model berhasil dilatih.")
    
    # --- Tahap D: Evaluasi Model ---
    print("\n--- Mengevaluasi performa model ---")
    predictions = model.predict(X_val_tfidf)
    rmse = np.sqrt(mean_squared_error(y_val, predictions))
    
    end_time = time.time()
    duration = end_time - start_time
    
    print("\n==============================================")
    print(f" 🎯 SKOR FINAL RMSE: {rmse:.4f}")
    print(f" ⏱️  Waktu Eksekusi: {duration:.2f} detik")
    print("==============================================")

Mempersiapkan data...
Melakukan vektorisasi teks dengan TF-IDF...
✅ Data fitur siap.

--- Melatih model LightGBM ---
✅ Model berhasil dilatih.

--- Mengevaluasi performa model ---


c:\KULIAH STATISTIKA UNIVERSITAS SYIAH KUALA\Envi_3.10.8\lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(



 🎯 SKOR FINAL RMSE: 16.5242
 ⏱️  Waktu Eksekusi: 812.97 detik


In [ ]:
import pandas as pd
import os
import re
import time
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
import lightgbm as lgb

# =============================================================================
# 1. KONFIGURASI
# =============================================================================
DATA_PATH = 'objective-quest-2025/file_putusan/'
TRAIN_CSV_PATH = 'train.csv'
TEST_CSV_PATH = 'test.csv'
SUBMISSION_PATH = 'submission_kfold_ensemble.csv'
RANDOM_STATE = 42
N_SPLITS = 5 # Jumlah fold untuk cross-validation

# =============================================================================
# 2. FUNGSI-FUNGSI HELPER (Golden Baseline Version)
# =============================================================================
def clean_text_best_performing(text):
    text = text.lower()
    noise_patterns = [
        re.compile(r'^\s*hkama.*$', re.MULTILINE), re.compile(r'^\s*ahkamah agung repub.*$', re.MULTILINE),
        re.compile(r'^\s*mah agung republik indonesia.*$', re.MULTILINE), re.compile(r'^\s*blik indonesi.*$', re.MULTILINE),
        re.compile(r'^\s*direktori putusan.*$', re.MULTILINE), re.compile(r'^\s*disclaimer\s*$', re.MULTILINE),
        re.compile(r'^\s*kepaniteraan mahkamah agung.*$', re.MULTILINE), re.compile(r'^\s*halaman \d+ dari \d+.*$', re.MULTILINE),
        re.compile(r'.*\(ext\.\d+\)$', re.MULTILINE)
    ]
    for pattern in noise_patterns:
        text = pattern.sub('', text)
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def get_cleaned_text_robust(doc_id):
    file_path = os.path.join(DATA_PATH, f'{doc_id}.txt')
    try:
        with open(file_path, 'r', encoding='utf-8', errors='ignore') as file:
            return clean_text_best_performing(file.read())
    except FileNotFoundError:
        return ""

# =============================================================================
# 3. EKSEKUSI UTAMA - K-FOLD & ENSEMBLE
# =============================================================================
if __name__ == '__main__':
    start_time = time.time()
    
    # --- Tahap A: Persiapan Data Lengkap ---
    print("Mempersiapkan data training...")
    df_train = pd.read_csv(TRAIN_CSV_PATH)
    df_train['teks'] = df_train['id'].apply(get_cleaned_text_robust)
    df_train.dropna(subset=['teks'], inplace=True)
    df_train = df_train[df_train['teks'].str.len() > 0].copy()
    df_train = df_train[df_train['lama hukuman (bulan)'] != 88888].copy().reset_index(drop=True)
    
    X = df_train['teks']
    y = df_train['lama hukuman (bulan)']
    print("✅ Data training siap.")

    # --- Tahap B: Vektorisasi Teks (Dilakukan sekali) ---
    print("\nMelakukan vektorisasi pada seluruh teks training...")
    vectorizer = TfidfVectorizer(max_features=15000, min_df=3, ngram_range=(1, 2))
    X_tfidf_full = vectorizer.fit_transform(X)
    print("✅ Vektorisasi selesai.")

    # --- Tahap C: K-Fold Cross-Validation & Training ---
    kf = KFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
    oof_predictions = np.zeros(X_tfidf_full.shape[0])
    trained_models = [] # List untuk menyimpan 5 model kita

    print(f"\n--- Memulai {N_SPLITS}-Fold Cross-Validation ---")
    for fold, (train_index, val_index) in enumerate(kf.split(X_tfidf_full)):
        print(f"\n--- Melatih Fold {fold + 1}/{N_SPLITS} ---")
        X_train, X_val = X_tfidf_full[train_index], X_tfidf_full[val_index]
        y_train, y_val = y.iloc[train_index], y.iloc[val_index]

        lgb_params = {'objective': 'rmse', 'metric': 'rmse', 'n_estimators': 1000, 'learning_rate': 0.05,
                      'random_state': RANDOM_STATE + fold, 'n_jobs': -1}
        model = lgb.LGBMRegressor(**lgb_params)
        model.fit(X_train, y_train, eval_set=[(X_val, y_val)], callbacks=[lgb.early_stopping(100, verbose=False)])

        preds = model.predict(X_val)
        oof_predictions[val_index] = preds
        trained_models.append(model) # Simpan model yang sudah dilatih
        print(f"✅ Fold {fold + 1} RMSE: {np.sqrt(mean_squared_error(y_val, preds)):.4f}")

    # --- Tahap D: Evaluasi Akhir (OOF) ---
    overall_rmse = np.sqrt(mean_squared_error(y, oof_predictions))
    print("\n==============================================")
    print(f"🎯 SKOR RMSE K-FOLD KESELURUHAN: {overall_rmse:.4f}")
    print("==============================================")
    
    # --- Tahap E: Prediksi pada Data Tes Menggunakan Ensemble ---
    print("\n--- Memproses data tes untuk prediksi ensemble ---")
    df_test = pd.read_csv(TEST_CSV_PATH)
    df_test['teks'] = df_test['id'].apply(get_cleaned_text_robust)
    X_test_tfidf = vectorizer.transform(df_test['teks'])
    
    test_predictions = []
    for model in trained_models:
        preds = model.predict(X_test_tfidf)
        test_predictions.append(preds)
        
    # Ambil rata-rata dari prediksi kelima model
    final_predictions = np.mean(test_predictions, axis=0)
    final_predictions[final_predictions < 0] = 0
    print("✅ Prediksi ensemble selesai.")
    
    # --- Tahap F: Membuat File Submisi ---
    print(f"\nMembuat file submisi: {SUBMISSION_PATH}...")
    submission_df = pd.DataFrame({'id': df_test['id'], 'lama hukuman (bulan)': final_predictions})
    submission_df['lama hukuman (bulan)'] = submission_df['lama hukuman (bulan)'].round().astype(int)
    submission_df.to_csv(SUBMISSION_PATH, index=False)
    
    print(f"\n🎉 File '{SUBMISSION_PATH}' berhasil dibuat!")
    end_time = time.time()
    print(f"\nTotal waktu eksekusi: {end_time - start_time:.2f} detik.")

Mempersiapkan data training...
✅ Data training siap.

Melakukan vektorisasi pada seluruh teks training...
✅ Vektorisasi selesai.

--- Memulai 5-Fold Cross-Validation ---

--- Melatih Fold 1/5 ---
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 1.352743 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2689538
[LightGBM] [Info] Number of data points in the train set: 13254, number of used features: 14805
[LightGBM] [Info] Start training from score 39.958201


c:\KULIAH STATISTIKA UNIVERSITAS SYIAH KUALA\Envi_3.10.8\lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


✅ Fold 1 RMSE: 16.4922

--- Melatih Fold 2/5 ---
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 1.392759 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2684209
[LightGBM] [Info] Number of data points in the train set: 13254, number of used features: 14803
[LightGBM] [Info] Start training from score 40.082239


c:\KULIAH STATISTIKA UNIVERSITAS SYIAH KUALA\Envi_3.10.8\lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


✅ Fold 2 RMSE: 16.8943

--- Melatih Fold 3/5 ---
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 1.300961 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2686923
[LightGBM] [Info] Number of data points in the train set: 13254, number of used features: 14809
[LightGBM] [Info] Start training from score 39.916403


...

✅ Fold 4 RMSE: 16.7330

--- Melatih Fold 5/5 ---
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 1.255531 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2691489
[LightGBM] [Info] Number of data points in the train set: 13255, number of used features: 14811
[LightGBM] [Info] Start training from score 39.898604


c:\KULIAH STATISTIKA UNIVERSITAS SYIAH KUALA\Envi_3.10.8\lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


✅ Fold 5 RMSE: 15.2332

🎯 SKOR RMSE K-FOLD KESELURUHAN: 16.2386

--- Memproses data tes untuk prediksi ensemble ---


c:\KULIAH STATISTIKA UNIVERSITAS SYIAH KUALA\Envi_3.10.8\lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


✅ Prediksi ensemble selesai.

Membuat file submisi: submission_kfold_ensemble.csv...

🎉 File 'submission_kfold_ensemble.csv' berhasil dibuat!

Total waktu eksekusi: 3242.48 detik.


In [ ]:
# =============================================================================
# 4. PREDIKSI PADA DATA TES & PEMBUATAN SUBMISI
# =============================================================================
# Pastikan sel training di atas sudah dijalankan terlebih dahulu

TEST_CSV_PATH = 'test.csv'
SUBMISSION_CSV_PATH = 'submission.csv'

print("--- Memulai proses prediksi pada data tes ---")
start_pred_time = time.time()

# --- Tahap G: Memuat dan Membersihkan Data Tes ---
print(f"Membaca file {TEST_CSV_PATH}...")
df_test = pd.read_csv(TEST_CSV_PATH)

print("Membersihkan teks data tes menggunakan fungsi yang sama...")
# Menerapkan fungsi yang sama persis seperti pada data training
df_test['teks'] = df_test['id'].apply(get_cleaned_text_robust)

# --- Tahap H: Vektorisasi Teks Tes ---
print("Melakukan vektorisasi pada teks data tes...")
# PENTING: Gunakan .transform() BUKAN .fit_transform()
# Ini untuk memastikan data tes diproses dengan "kamus" yang sama dari data training
X_test_tfidf = vectorizer.transform(df_test['teks'])
print("✅ Teks tes berhasil divektorisasi.")

# --- Tahap I: Membuat Prediksi ---
print("Membuat prediksi menggunakan model yang sudah dilatih...")
predictions = model.predict(X_test_tfidf)

# Post-processing: Pastikan tidak ada prediksi hukuman negatif
predictions[predictions < 0] = 0
print("✅ Prediksi selesai.")

# --- Tahap J: Membuat File Submisi ---
print(f"\nMembuat file submisi: {SUBMISSION_CSV_PATH}...")
submission_df = pd.DataFrame({'id': df_test['id'], 'lama hukuman (bulan)': predictions})

# Membulatkan hasil prediksi ke integer terdekat jika diinginkan
submission_df['lama hukuman (bulan)'] = submission_df['lama hukuman (bulan)'].round().astype(int)

submission_df.to_csv(SUBMISSION_CSV_PATH, index=False)

end_pred_time = time.time()
print("\n==============================================")
print(f"🎉 File '{SUBMISSION_CSV_PATH}' berhasil dibuat!")
print("==============================================")
print("Contoh isi file submisi:")
print(submission_df.head().to_string(index=False))
print(f"\nWaktu eksekusi untuk prediksi: {end_pred_time - start_pred_time:.2f} detik.")

--- Memulai proses prediksi pada data tes ---
Membaca file test.csv...
Membersihkan teks data tes menggunakan fungsi yang sama...
Melakukan vektorisasi pada teks data tes...
✅ Teks tes berhasil divektorisasi.
Membuat prediksi menggunakan model yang sudah dilatih...


c:\KULIAH STATISTIKA UNIVERSITAS SYIAH KUALA\Envi_3.10.8\lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


✅ Prediksi selesai.

Membuat file submisi: submission.csv...

🎉 File 'submission.csv' berhasil dibuat!
Contoh isi file submisi:
       id  lama hukuman (bulan)
 doc_7650                    12
doc_22122                   112
doc_13812                    59
doc_16571                    24
 doc_9900                     7

Waktu eksekusi untuk prediksi: 184.48 detik.
